In [1]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [3]:
# Dataset statistics before tokenization
emotions_unique = df["emotion"].unique()
print("Dataset statistics before tokenization for each unique emotion:")
print()
for emotion in emotions_unique:
    subset = df[df["emotion"] == emotion]
    N = len(subset["emotion"])
    print(f"{emotion} has {N} occurences, {round(N/len(df["emotion"]) * 100, 2)}%")
    average_length = subset["content"].str.len().mean()
    print(f"{emotion} has an average length of {average_length:.2f} characters")
    print()


Dataset statistics before tokenization for each unique emotion:

joy has 1092 occurences, 14.53%
joy has an average length of 103.14 characters

fear has 1076 occurences, 14.32%
fear has an average length of 123.96 characters

anger has 1079 occurences, 14.36%
anger has an average length of 130.65 characters

sadness has 1082 occurences, 14.4%
sadness has an average length of 103.93 characters

disgust has 1066 occurences, 14.18%
disgust has an average length of 114.61 characters

shame has 1071 occurences, 14.25%
shame has an average length of 116.73 characters

guilt has 1050 occurences, 13.97%
guilt has an average length of 124.89 characters



In [4]:
# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


In [5]:
df.head(5)
# print avarage token length of sentence here.

,emotion,content,tokens
0,joy,On days when I feel close to my partner and ot...,"[On, days, when, I, feel, close, to, my, partn..."
1,fear,Every time I imagine that someone I love or I ...,"[Every, time, I, imagine, that, someone, I, lo..."
2,anger,When I had been obviously unjustly treated and...,"[When, I, had, been, obviously, unjustly, trea..."
3,sadness,When I think about the short time that we live...,"[When, I, think, about, the, short, time, that..."
4,disgust,At a gathering I found myself involuntarily si...,"[At, a, gathering, I, found, myself, involunta..."


In [6]:
# Representation 2: Bag of words
vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)

bag_matrix = vectorizer.fit_transform(df["tokens"].tolist())

bag_df = pd.DataFrame(bag_matrix.toarray(),
                      columns=vectorizer.get_feature_names_out(),
                      index=df.index
)

print(bag_df.head(5))

   !  $  %  &  '  ''  'd  'll  'm  're  ...  yr.  yrs  yukky  zero  \
0  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
1  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
2  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
3  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
4  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   

   zig-zagging  zip  zipper  zombies  zone  zoophiliac  
0            0    0       0        0     0           0  
1            0    0       0        0     0           0  
2            0    0       0        0     0           0  
3            0    0       0        0     0           0  
4            0    0       0        0     0           0  

[5 rows x 10173 columns]


In [7]:
# Representation 3: Bigrams

bigrams = []
for sentence in df["tokens"]:
    bigrams_per_sentence = []
    for i in range(len(sentence) - 1):
        bigram = sentence[i] + " " + sentence[i+1]
        bigrams_per_sentence.append(bigram)
    bigrams.append(bigrams_per_sentence)
df["bigrams"] = bigrams

In [8]:
# Representation 4: Trigrams

trigrams = []
for sentence in df["tokens"]:
    trigrams_per_sentence = []
    for i in range(len(sentence) - 2):
        trigram = sentence[i] + " " + sentence[i+1] + " " + sentence[i+2]
        trigrams_per_sentence.append(trigram)
    trigrams.append(trigrams_per_sentence)
df["trigrams"] = trigrams

In [9]:
print(df["trigrams"][0])

['On days when', 'days when I', 'when I feel', 'I feel close', 'feel close to', 'close to my', 'to my partner', 'my partner and', 'partner and other', 'and other friends', 'other friends .', 'friends . When', '. When I', 'When I feel', 'I feel at', 'feel at peace', 'at peace with', 'peace with myself', 'with myself and', 'myself and also', 'and also experience', 'also experience a', 'experience a close', 'a close contact', 'close contact with', 'contact with people', 'with people whom', 'people whom I', 'whom I regard', 'I regard greatly', 'regard greatly .']


In [10]:
# Architecture 1: Log-Linear
import itertools
nltk.download("punkt_tab")

corpus = " ".join(df["content"].fillna("").astype(str))

vocab = sorted(set(word_tokenize(corpus)))
token_to_id = {token: i for i, token in enumerate(vocab)}

class Encoder:
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# for seed in my_seeds:  
#     my_cross_validation_sets.append(
#         cross_validation_split(
#             # we pair the texts with their labels
#             pairs=list(zip(dataset["train"]["text"], dataset["train"]["label"])), 
#             k=num_folds,
#             seed=seed
#         )
#     )

class TokenIndicator(Encoder):
    def __init__(self, token_to_id):
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.token_to_id)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self), dtype=np.float32)

        for token in set(word_tokenize(x)):
            token_id = self.token_to_id.get(token)
            if token_id is not None:
                vector[token_id] = 1

        return vector
    
token_indicator_encoder = TokenIndicator(token_to_id)
doc_repr = token_indicator_encoder(corpus)

print(len(doc_repr))
print(doc_repr.sum())

[nltk_data] Downloading package punkt_tab to /home/colin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


10176
10176.0


In [22]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [23]:
# Architecture 2: Single hidden layer MLP
import smlp

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    metrics, best_epoch, best_pred = smlp.train_neural_network_model(
    cv[fold]["training"],
    cv[fold]["validation"],
    num_classes=len(labels),
    encoder=token_indicator_encoder,
    save_to=f"{results_folder}/mlp_{fold}_test",
    num_epochs=10)
    accuracies.append(best_pred)
print(accuracies)

Transforming documents...
Document transformation done! Time taken: 2.226 seconds
Dataloader done! Time taken: 0.562 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7991 | train acc: 0.3925 | val loss: 1.5662 | val acc: 0.5146
Epoch 02 | train loss: 1.1711 | train acc: 0.6766 | val loss: 1.2392 | val acc: 0.5672
Epoch 03 | train loss: 0.7391 | train acc: 0.8001 | val loss: 1.1744 | val acc: 0.5778
Epoch 04 | train loss: 0.5041 | train acc: 0.8716 | val loss: 1.2086 | val acc: 0.5758
Epoch 05 | train loss: 0.3615 | train acc: 0.9143 | val loss: 1.2498 | val acc: 0.5698
Epoch 06 | train loss: 0.2687 | train acc: 0.9383 | val loss: 1.3079 | val acc: 0.5738
Epoch 07 | train loss: 0.2057 | train acc: 0.9579 | val loss: 1.3809 | val acc: 0.5618
Epoch 08 | train loss: 0.1591 | train acc: 0.9692 | val loss: 1.4506 | val acc: 0.5658
Epoch 09 | train loss: 0.1248 | train acc: 0.9782 | val loss: 1.5373 | val acc: 0.5632
Epoch 10 | train loss: 0.1032 | train acc: 0.9815 | val loss: 1.6074 | val acc: 0.5652
Training done! Time taken: 15.654 seconds
Transforming documents...
Document transformation done! Time taken: 2.257 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8120 | train acc: 0.3943 | val loss: 1.5725 | val acc: 0.5156
Epoch 02 | train loss: 1.1932 | train acc: 0.6730 | val loss: 1.2292 | val acc: 0.5842
Epoch 03 | train loss: 0.7492 | train acc: 0.7983 | val loss: 1.1781 | val acc: 0.5782
Epoch 04 | train loss: 0.5113 | train acc: 0.8673 | val loss: 1.1852 | val acc: 0.5828
Epoch 05 | train loss: 0.3665 | train acc: 0.9135 | val loss: 1.2311 | val acc: 0.5782
Epoch 06 | train loss: 0.2730 | train acc: 0.9370 | val loss: 1.2881 | val acc: 0.5762
Epoch 07 | train loss: 0.2087 | train acc: 0.9559 | val loss: 1.3586 | val acc: 0.5689
Epoch 08 | train loss: 0.1642 | train acc: 0.9662 | val loss: 1.4173 | val acc: 0.5722
Epoch 09 | train loss: 0.1313 | train acc: 0.9746 | val loss: 1.4784 | val acc: 0.5662
Epoch 10 | train loss: 0.1067 | train acc: 0.9815 | val loss: 1.5428 | val acc: 0.5695
Training done! Time taken: 15.448 seconds
Transforming documents...
Document transformation done! Time taken: 2.323 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8041 | train acc: 0.3890 | val loss: 1.5693 | val acc: 0.5230
Epoch 02 | train loss: 1.1805 | train acc: 0.6714 | val loss: 1.2298 | val acc: 0.5835
Epoch 03 | train loss: 0.7439 | train acc: 0.8029 | val loss: 1.1648 | val acc: 0.5928
Epoch 04 | train loss: 0.5111 | train acc: 0.8741 | val loss: 1.1911 | val acc: 0.5921
Epoch 05 | train loss: 0.3645 | train acc: 0.9165 | val loss: 1.2418 | val acc: 0.5928
Epoch 06 | train loss: 0.2665 | train acc: 0.9443 | val loss: 1.3043 | val acc: 0.5842
Epoch 07 | train loss: 0.2061 | train acc: 0.9586 | val loss: 1.3776 | val acc: 0.5802
Epoch 08 | train loss: 0.1574 | train acc: 0.9706 | val loss: 1.4541 | val acc: 0.5768
Epoch 09 | train loss: 0.1255 | train acc: 0.9777 | val loss: 1.5301 | val acc: 0.5762
Epoch 10 | train loss: 0.0992 | train acc: 0.9845 | val loss: 1.6033 | val acc: 0.5709
Training done! Time taken: 15.430 seconds
Transforming documents...
Document transformation done! Time taken: 2.283 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7998 | train acc: 0.3873 | val loss: 1.5575 | val acc: 0.5336
Epoch 02 | train loss: 1.1677 | train acc: 0.6797 | val loss: 1.2313 | val acc: 0.5921
Epoch 03 | train loss: 0.7392 | train acc: 0.7978 | val loss: 1.1672 | val acc: 0.5928
Epoch 04 | train loss: 0.5023 | train acc: 0.8741 | val loss: 1.1822 | val acc: 0.5882
Epoch 05 | train loss: 0.3582 | train acc: 0.9155 | val loss: 1.2262 | val acc: 0.5788
Epoch 06 | train loss: 0.2653 | train acc: 0.9431 | val loss: 1.2956 | val acc: 0.5735
Epoch 07 | train loss: 0.2023 | train acc: 0.9589 | val loss: 1.3676 | val acc: 0.5722
Epoch 08 | train loss: 0.1576 | train acc: 0.9702 | val loss: 1.4278 | val acc: 0.5695
Epoch 09 | train loss: 0.1247 | train acc: 0.9779 | val loss: 1.5047 | val acc: 0.5675
Epoch 10 | train loss: 0.1020 | train acc: 0.9840 | val loss: 1.5735 | val acc: 0.5589
Training done! Time taken: 15.127 seconds
Transforming documents...
Document transformation done! Time taken: 2.153 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8008 | train acc: 0.3963 | val loss: 1.5415 | val acc: 0.5449
Epoch 02 | train loss: 1.1772 | train acc: 0.6694 | val loss: 1.1984 | val acc: 0.5961
Epoch 03 | train loss: 0.7492 | train acc: 0.7934 | val loss: 1.1227 | val acc: 0.6128
Epoch 04 | train loss: 0.5139 | train acc: 0.8714 | val loss: 1.1412 | val acc: 0.6015
Epoch 05 | train loss: 0.3700 | train acc: 0.9084 | val loss: 1.1850 | val acc: 0.5888
Epoch 06 | train loss: 0.2760 | train acc: 0.9375 | val loss: 1.2355 | val acc: 0.5842
Epoch 07 | train loss: 0.2122 | train acc: 0.9546 | val loss: 1.3000 | val acc: 0.5835
Epoch 08 | train loss: 0.1662 | train acc: 0.9682 | val loss: 1.3730 | val acc: 0.5735
Epoch 09 | train loss: 0.1345 | train acc: 0.9742 | val loss: 1.4279 | val acc: 0.5635
Epoch 10 | train loss: 0.1080 | train acc: 0.9814 | val loss: 1.5069 | val acc: 0.5602
Training done! Time taken: 14.960 seconds
[array([1, 2, 1, ..., 3, 5, 0], shape=(1504,)), array([6, 2, 2, ..., 5, 0, 4], shape=(15